У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [104]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
import os
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import pandas as pd
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import operator
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTENC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, precision_score, recall_score

In [106]:
customers_df = pd.read_csv('customer_segmentation_train.csv', index_col=0)
cus_df = customers_df.dropna(subset=['Profession'])
cus_df = cus_df.dropna(subset=['Graduated'])
cus_df = cus_df.dropna(subset=['Var_1'])
cus_df.isnull().sum()

Gender               0
Ever_Married       131
Age                  0
Graduated            0
Profession           0
Work_Experience    784
Spending_Score       0
Family_Size        302
Var_1                0
Segmentation         0
dtype: int64

In [108]:
customers_df.describe()

,Age,Work_Experience,Family_Size
count,8068.000000,7239.000000,7733.000000
mean,43.466906,2.641663,2.850123
std,16.711696,3.406763,1.531413
min,18.000000,0.000000,1.000000
25%,30.000000,0.000000,2.000000
50%,40.000000,1.000000,3.000000
75%,53.000000,4.000000,4.000000
max,89.000000,14.000000,9.000000


In [110]:
for col in customers_df.select_dtypes(include='object').columns:
    print(f'\n{col}:\n{customers_df[col].value_counts()}')


Gender:
Gender
Male      4417
Female    3651
Name: count, dtype: int64

Ever_Married:
Ever_Married
Yes    4643
No     3285
Name: count, dtype: int64

Graduated:
Graduated
Yes    4968
No     3022
Name: count, dtype: int64

Profession:
Profession
Artist           2516
Healthcare       1332
Entertainment     949
Engineer          699
Doctor            688
Lawyer            623
Executive         599
Marketing         292
Homemaker         246
Name: count, dtype: int64

Spending_Score:
Spending_Score
Low        4878
Average    1974
High       1216
Name: count, dtype: int64

Var_1:
Var_1
Cat_6    5238
Cat_4    1089
Cat_3     822
Cat_2     422
Cat_7     203
Cat_1     133
Cat_5      85
Name: count, dtype: int64

Segmentation:
Segmentation
D    2268
A    1972
C    1970
B    1858
Name: count, dtype: int64


In [112]:
cus_df.columns.tolist()

['Gender',
 'Ever_Married',
 'Age',
 'Graduated',
 'Profession',
 'Work_Experience',
 'Spending_Score',
 'Family_Size',
 'Var_1',
 'Segmentation']

In [114]:
train_df, val_df = train_test_split(cus_df, test_size=0.20, random_state=42, stratify=cus_df["Segmentation"])

input_cols = ['Gender', 'Ever_Married', 'Age', 'Graduated', 'Profession', 'Work_Experience', 'Spending_Score', 'Family_Size', 'Var_1']
target_col = 'Segmentation'
train_inputs, train_targets = train_df[input_cols].copy(), train_df[target_col].copy()
val_inputs, val_targets = val_df[input_cols].copy(), val_df[target_col].copy()

numeric_cols = train_inputs[input_cols].select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs[input_cols].select_dtypes('object').columns.tolist()
cat_indices = [train_inputs.columns.get_loc(col) for col in categorical_cols]

num_imputer = SimpleImputer(strategy='median')
train_inputs[['Work_Experience', 'Family_Size']] = num_imputer.fit_transform(train_inputs[['Work_Experience', 'Family_Size']])
val_inputs[['Work_Experience', 'Family_Size']] = num_imputer.transform(val_inputs[['Work_Experience', 'Family_Size']])

cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
train_inputs[["Ever_Married"]] = cat_imputer.fit_transform(train_inputs[["Ever_Married"]])
val_inputs[["Ever_Married"]] = cat_imputer.transform(val_inputs[["Ever_Married"]])

numeric_transformer = Pipeline(steps=[
    ('scaler', MinMaxScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [124]:
SMOTENC_pipeline = Pipeline(steps=[
    ('sampler', SMOTENC(categorical_features=cat_indices, random_state=42)),
    ('preprocessor', preprocessor),
    ('classifier',  OneVsRestClassifier(LogisticRegression(solver='liblinear')))
])

SMOTETomek_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('sampler', SMOTETomek(random_state=0)),
    ('classifier',  OneVsRestClassifier(LogisticRegression(solver='liblinear')))
])

origin_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',  OneVsRestClassifier(LogisticRegression(solver='liblinear')))
])

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [126]:
SMOTENC_pipeline.fit(train_inputs, train_targets)
SMOTETomek_pipeline.fit(train_inputs, train_targets)
origin_pipeline.fit(train_inputs, train_targets)

origin_preds = origin_pipeline.predict(val_inputs)
print(classification_report(val_targets, origin_preds))

SMOTENC_preds = SMOTENC_pipeline.predict(val_inputs)
print(classification_report(val_targets, SMOTENC_preds))

SMOTETomek_preds = SMOTETomek_pipeline.predict(val_inputs)
print(classification_report(val_targets, SMOTETomek_preds))

              precision    recall  f1-score   support

           A       0.41      0.47      0.44       380
           B       0.44      0.13      0.20       361
           C       0.48      0.66      0.56       384
           D       0.65      0.73      0.69       435

    accuracy                           0.51      1560
   macro avg       0.50      0.50      0.47      1560
weighted avg       0.50      0.51      0.48      1560

              precision    recall  f1-score   support

           A       0.41      0.50      0.45       380
           B       0.41      0.20      0.27       361
           C       0.49      0.61      0.54       384
           D       0.68      0.70      0.69       435

    accuracy                           0.51      1560
   macro avg       0.50      0.50      0.49      1560
weighted avg       0.51      0.51      0.50      1560

              precision    recall  f1-score   support

           A       0.42      0.48      0.45       380
           B       0.

Я би обрала для оцінки цих моделей f1-score, як би воно не було майже однаковим :)
Моделі з балансуванням трошки кращі, але між ними немає різниці між моделями тому що цільові класи досить сбалансовані, тому майже немає різниці яким способом ми їх балансуєм